# DEmod model training on a free GPU (Colab or Kaggle)

Trains SpecCFO, DemodAMC and the recreated paper baselines on freshly simulated signals, then runs the same evaluation as `research/README.md`.
Nothing here uses real RF captures.

**Colab:** Runtime, Change runtime type, GPU (T4). **Kaggle:** Settings, Accelerator, GPU T4 x2 (turn Internet on).
Free sessions can drop. Checkpoints go to Google Drive on Colab, and every run resumes from its last checkpoint, so just re-run the cells.

In [ ]:
import subprocess, sys, os
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout or 'NO GPU: change the runtime type first')
import jax; print('JAX devices:', jax.devices())

In [ ]:
# Colab ships JAX with CUDA. On Kaggle or a fresh machine, uncomment the next line:
# !pip install -q -U "jax[cuda12]"
!pip install -q optax matplotlib

In [ ]:
# Get the code. Option A: clone the repo (after you pushed the patch). Option B: upload demod-research.zip.
REPO = 'https://github.com/richennacht/demod.git'
if not os.path.exists('demod'):
    r = subprocess.run(['git','clone','--depth','1',REPO],capture_output=True,text=True); print(r.stdout, r.stderr)
if not os.path.exists('demod/research/train_cfo.py'):
    from google.colab import files  # Option B
    up = files.upload(); name = next(iter(up)); !unzip -q -o {name} -d demod
%cd demod

In [ ]:
# Keep checkpoints on Drive (Colab) so a dropped session resumes. Skipped on Kaggle.
try:
    from google.colab import drive; drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/demod_ckpt research
    !rm -rf research/checkpoints && ln -s /content/drive/MyDrive/demod_ckpt research/checkpoints
except Exception as e:
    print('no Drive, using local disk:', e)

In [ ]:
# Full-budget GPU run. CPU runs in the repo used CFO_STEPS=3000, AMC_STEPS=2000, BATCH=64.
# A T4 handles BATCH=256 comfortably. Expect roughly an hour or two in total.
import os
os.environ.update(CFO_STEPS='20000', AMC_STEPS='12000', BATCH='256')
!bash research/run_all_training.sh

In [ ]:
!python research/eval_cfo.py
!python research/eval_amc.py
!python research/make_report.py

In [ ]:
# Download the trained models and the result tables.
!mkdir -p out && cp -r data/models out/ && cp -r research/results out/
!zip -qr demod-trained.zip out
from google.colab import files; files.download('demod-trained.zip')

Copy `data/models/*.npz` and `research/results/*` into the repo, commit, and push. The API loads `data/models/speccfo.npz` and `data/models/demod_amc.npz` when present and falls back to the old models when absent.